# 171Yb+: hyperfine structure, Breit-Rabi, and sideband coupling

A physics-first path from atomic hyperfine structure to motional sideband coupling.

171Yb+ is a common trapped-ion qubit species: nuclear spin $I=1/2$, ground term $^2S_{1/2}$ ($J=1/2$), giving
a simple two-level hyperfine manifold ($F=0,1$) at zero field that mixes and Zeeman-shifts at finite field
(the standard Breit-Rabi problem). We build that structure with `hyperfine_levels`, then switch to the
*motional* side of the same ion and compute sideband coupling strengths with `trap.py`.

Two disclaimers up front: the hyperfine constant `A_hf` below is the commonly-cited literature value for
171Yb+'s ground-state clock splitting; the nuclear g-factor `g_I` is order-of-magnitude illustrative only
(check a primary source before using either for real analysis). Neither claim is metrology-grade -- the
point of this notebook is the workflow, not the tenth decimal place.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

import htdse as ht

## 1. Ground-state hyperfine structure at zero field

$I=1/2$, $J=1/2$ (a pure spin term, $L=0$) means `hyperfine_matrix`'s only nonzero hyperfine constant is the
magnetic-dipole term `A` -- no electric quadrupole (`B`) is possible below $I,J>1/2$. `g_J` is worth computing
rather than assuming: with $L=0$, the electron's orbital g-factor $g_l$ shouldn't contribute at all, and
`g_sum` (ported straight from AMO.jl) confirms it -- $g_J$ comes out equal to the free-electron $g_s$ to
every digit, since coupling in zero orbital angular momentum is a no-op.

In [ ]:
I, J = 0.5, 0.5
A_hf = 12.642812e9  # Hz -- commonly cited 171Yb+ ground-state hyperfine splitting (~12.6428 GHz)

g_J = ht.g_sum(J, 0.0, 1.0, 0.5, ht.g_s)  # J built from L=0 coupled with S=1/2
print(f"g_J = {g_J} (== g_s = {ht.g_s}: L=0 contributes nothing, as expected)")

evals0, _ = ht.hyperfine_levels(I, J, Bm=0.0, Ahf=A_hf)
print("levels at B=0 (Hz):", evals0)
splitting = evals0.max() - evals0.min()
print(f"F=1 <-> F=0 splitting: {splitting/1e9:.6f} GHz")

## 2. Breit-Rabi diagram

Turning on a field mixes different-$F$, same-$m_F$ states (`hyperfine_matrix`'s off-diagonal blocks) --
the eigenvalues bend away from their zero-field values, linearly at first (Zeeman) and then curving as the
field competes with the hyperfine coupling. This is exactly the diagonalization `hyperfine_levels` exists
for (AMO.jl builds the matrix but stops there).

In [ ]:
g_I = -4.7e-4  # illustrative only -- nuclear g-factor, suppressed by ~m_e/m_p relative to g_J
Bms = np.linspace(0, 0.01, 200)  # arbitrary field-strength units, chosen so g*Bm is a few GHz over the range
levels = np.array([ht.hyperfine_levels(I, J, Bm=Bm, g_I=g_I, g_J=g_J, Ahf=A_hf)[0] for Bm in Bms])

fig, ax = plt.subplots(figsize=(5, 4))
for i in range(levels.shape[1]):
    ax.plot(Bms, levels[:, i] / 1e9)
ax.set_xlabel("B (arb. units)")
ax.set_ylabel("Energy (GHz)")
ax.set_title("171Yb+ ground-state Breit-Rabi diagram")
plt.show()

## 3. Sideband coupling for a real 171Yb+ mass

Switching to the motional side: `lamb_dicke` turns a real trap frequency and laser wavelength into the
dimensionless $\eta$ that `trap.sideband`/`IonChain.sideband_coupling` then consume. Using the actual
171Yb+ mass here is a nice coincidence check -- it lands almost exactly on the mass AMO.jl's own test suite
used for its `\eta` regression fixture (`2.84e-25 kg`), which is presumably no accident on the original
author's part.

In [ ]:
amu = 1.66053906660e-27
mass_yb171 = 171 * amu
print(f"mass_yb171 = {mass_yb171:.4e} kg  (AMO.jl's own test fixture: 2.84e-25 kg)")

fm = 1e6            # 1 MHz axial trap frequency
lam = 369.5e-9      # 2S1/2-2P1/2 Yb+ transition, commonly used for Doppler cooling/detection
eta = ht.lamb_dicke(mass_yb171, fm=fm, lambda_p=lam)
print(f"eta = {eta:.4f}")

chain = ht.ion_chain([ht.Mode(nu=fm, eta=eta, n_max=10, name="axial")], n_ions=1)
for n in range(4):
    print(f"  carrier <n={n}|exp(i eta (a+adag))|n={n}> = {chain.sideband_coupling('axial', n, n):.4f}")
print(f"  red sideband |<0|...|1>| = {abs(chain.sideband_coupling('axial', 0, 1)):.4f}")

## 4. Thermally-averaged sideband flopping

`IonChain.thermal_sideband` averages the blue-sideband Rabi flopping curve over an initial thermal
distribution of phonons -- the contrast loss visible below is the same physics as the earlier
`practice_answers.py` thermal-dephasing exercise, now built on the ported (and independently verified)
`trap.py` machinery instead of a hand-rolled density-matrix simulation.

In [ ]:
ts = np.linspace(0, 5, 200)
nbar = 0.5
flop = [chain.thermal_sideband("axial", 1, t, nbar=nbar) for t in ts]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ts, flop)
ax.set_xlabel("t (arb. units, Omega=1)")
ax.set_ylabel("excited-state population")
ax.set_title(f"blue-sideband flopping, nbar={nbar}")
plt.show()